# 01 · Unified pipeline — synthetic FF-HEDM walkthrough

`midas-pipeline` is the unified MIDAS HEDM orchestrator. Its design
thesis: **FF is the single-scan degeneracy of PF.** One orchestrator,
one CLI, two scan modes (`--scan-mode {ff,pf,auto}`). Each layer runs a
mode-dependent ordered list of stages (zip_convert → hkl → peakfit →
merge_overlaps → calc_radius → transforms → … → binning → indexing →
refinement → … → consolidation).

This notebook runs the orchestrator end-to-end on a **synthetic** Au
FF-HEDM dataset (no external data, CPU only), now **through indexing**.
We:

1. Forward-simulate a small synthetic single-detector dataset.
2. Drive `midas-pipeline run --scan-mode ff` through ingest → hkl →
   peakfit → merge → radius → transforms → **binning → indexing**, and
   inspect per-stage timings + artefacts.
3. Confirm `Data.bin` (the per-bin spot-index table) is populated and the
   indexer recovers grain solutions.
4. Show how `ScanGeometry.ff()` is literally `pf_uniform` with one scan
   position — the degeneracy the package is built around.

> **Runtime** ~1–1.5 min on CPU (the per-frame peak search dominates).

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS libomp guard

import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np

from midas_pipeline import __version__, ScanGeometry
print('midas-pipeline', __version__)

## 1. Forward-simulate a synthetic dataset

`midas-pipeline simulate` is a scaffold in 0.2.x, so for a self-contained
synthetic dataset we use the sibling `midas_ff_pipeline.testing`
generator (pure-Python: `midas_diffract.simulate_panel_zarrs` +
`midas_hkls`, no C forward-sim). It reads geometry / lattice / scan keys
from the bundled `FF_HEDM/Example/Parameters.txt` and writes a
`.MIDAS.zip` archive plus a local `Parameters.txt`.

We keep the grain count small (the pattern just needs enough spots to
exercise the stages).

In [ ]:
from midas_ff_pipeline.testing import generate_synthetic_dataset

MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or Path.home() / 'opt' / 'MIDAS')
PARAMS_TEMPLATE = MIDAS_HOME / 'FF_HEDM' / 'Example' / 'Parameters.txt'
assert PARAMS_TEMPLATE.exists(), f'template not found: {PARAMS_TEMPLATE}'

WORK = Path(tempfile.mkdtemp(prefix='midas_pipeline_ff_'))
t0 = time.time()
zarr = generate_synthetic_dataset(
    out_dir=WORK / 'sim',
    params_template=PARAMS_TEMPLATE,
    n_grains=20,
    n_cpus=4,
)
print(f'synthetic dataset in {time.time() - t0:.1f}s')
print('zarr  :', zarr)
print('params:', WORK / 'sim' / PARAMS_TEMPLATE.name)

## 2. Run the orchestrator (FF mode, through indexing)

We call the CLI exactly as a user would. `--scan-mode ff` selects the
single-scan stage order. We run ingest → hkl → peakfit → merge → radius
→ transforms → **binning → indexing**, and `--skip` only the
refinement/consolidation tail (those add little to a binning/indexing
walkthrough). The default indexer backend is the in-process
PyTorch indexer (`--indexer-backend python`).

In [ ]:
RESULT = WORK / 'run'
cmd = [
    sys.executable, '-m', 'midas_pipeline', 'run',
    '--scan-mode', 'ff',
    '--params', str(WORK / 'sim' / PARAMS_TEMPLATE.name),
    '--result', str(RESULT),
    '--zarr', str(zarr),
    '--n-cpus', '4',
    '--device', 'cpu',
    '--dtype', 'float64',
    '--indexer-backend', 'python',
    '--skip', 'refinement',
    '--skip', 'process_grains', '--skip', 'consolidation',
]
print('running:', ' '.join(cmd[2:]), '\n')
t0 = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
print(f'exit={proc.returncode}  ({time.time() - t0:.1f}s)')
# Show the tail of the orchestrator log.
print('\n'.join(proc.stderr.strip().splitlines()[-8:]))
assert proc.returncode == 0, proc.stderr[-2000:]

## 3. Per-stage status

`midas-pipeline status` reads the provenance store written under each
`LayerNr_*/` and reports per-stage completion + wall time. This is the
same machinery that powers `resume` (skip already-complete stages) and
`inspect`.

In [ ]:
status = subprocess.run(
    [sys.executable, '-m', 'midas_pipeline', 'status', str(RESULT)],
    capture_output=True, text=True,
)
print(status.stdout)

## 4. Binning + indexing artefacts

The binning stage writes `Spots.bin` (observed spots, lab frame) plus the
per-bin spot-index table `Data.bin` / `nData.bin`. `Data.bin` is now
**non-empty** — this is the table the indexer mmaps to gather candidate
spots per (ring, eta, omega) bin. The indexer then writes `IndexBest.bin`
(one record per seed) and `IndexBestFull.bin` (matched spot pairs).

In [ ]:
layer_dir = RESULT / 'LayerNr_1'
interesting = ['hkls.csv', 'InputAll.csv', 'Spots.bin', 'Data.bin', 'nData.bin',
               'paramstest.txt', 'IndexBest.bin', 'IndexBestFull.bin']
for name in interesting:
    p = layer_dir / name
    print(f'{name:34s} {"OK " if p.exists() else "-- "} '
          f'{p.stat().st_size if p.exists() else 0:>14,} bytes')

assert (layer_dir / 'Data.bin').stat().st_size > 0, 'Data.bin must be non-empty'

## 5. Grain solutions

`IndexBest.bin` is a flat `float64` array of 15-column records, one per
indexed seed. Column 14 is `nMatches` (the count of matched predicted
spots); a seed with `nMatches > 0` produced an orientation solution.

In [ ]:
ib = np.fromfile(layer_dir / 'IndexBest.bin', dtype=np.float64)
assert ib.size % 15 == 0, ib.size
ib = ib.reshape(-1, 15)
n_seeds = ib.shape[0]
n_solved = int((ib[:, 14] > 0).sum())
print(f'seeds attempted : {n_seeds}')
print(f'seeds w/ a solution (nMatches > 0): {n_solved}')
print(f'best nMatches   : {ib[:, 14].max():.0f}')
assert n_solved > 0, 'expected at least one indexed seed'

## 6. FF is the single-scan degeneracy of PF

The package's central abstraction is `ScanGeometry`. FF is not a
separate code path conceptually — it is `pf_uniform` with exactly one
scan position at Y = 0. The orchestrator dispatches stages on
`scan_mode`, but the geometry object makes the degeneracy explicit.

In [ ]:
ff = ScanGeometry.ff()
pf = ScanGeometry.pf_uniform(n_scans=5, scan_step_um=2.0, beam_size_um=4.0)
print('FF :', ff)
print('PF :', pf)
print()
print('FF n_scans          :', ff.n_scans, '  (single scan position)')
print('FF scan_positions   :', ff.scan_positions)
print('PF n_scans          :', pf.n_scans)
print('PF scan_positions   :', pf.scan_positions, 'um')

## What just happened

The unified orchestrator ran the FF pipeline end-to-end **through
indexing** — frame ingest, HKL generation, per-frame peak fitting,
overlap merge, radius assignment, lab-frame transforms, binning, and
orientation indexing — all in-process via the shared `midas-*` kernel
packages, then reported provenance through `status`.

The remaining notebooks build on this working indexing stage:

| Notebook | Topic |
| --- | --- |
| 02 | indexer backend selector (`python` vs `c-omp`) — they agree |
| 03 | V-map per-spot relative volume + soft beam attribution |
| 04 | FF as the single-scan degeneracy of PF |

In [ ]:
# Tidy up the scratch workspace.
shutil.rmtree(WORK, ignore_errors=True)
print('cleaned', WORK)